# S4_1 — Setup and Architecture Verification
Verifies environment, loads the Meta I-JEPA checkpoint, initialises context/target encoders,
and builds the predictor. Run this before S4_3 to catch setup issues early.

Outputs:
- Architecture summary (parameters, layer groups, frozen layers)
- Forward pass shape verification
- VRAM estimate at training batch size


In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import torch
import timm

from stage4_leaf_jepa_pretraining.config_stage4 import *
from stage4_leaf_jepa_pretraining.pretrain_utils import (
    set_seed, get_layerwise_optimizer, IJEPAPredictor
)

set_seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU — training will be very slow.")
print(f"PyTorch: {torch.__version__}  |  timm: {timm.__version__}")
verify_config()


In [ ]:
# ── Cell 2: Load context encoder from Meta I-JEPA checkpoint ─────────────────
# CRITICAL: use global_pool='' to get per-patch tokens (not pooled)
print("Loading context encoder (ViT-H/14)...")
context_encoder = timm.create_model(
    VIT_MODEL_NAME, pretrained=False, num_classes=0, global_pool=""
)

ckpt = torch.load(IJEPA_CHECKPOINT, map_location="cpu")
if "target_encoder" in ckpt:
    state_dict = ckpt["target_encoder"]
    print("  Loaded from 'target_encoder' key (EMA encoder — correct)")
elif "encoder" in ckpt:
    state_dict = ckpt["encoder"]
    print("  Loaded from 'encoder' key")
else:
    state_dict = ckpt
    print("  Loaded from root checkpoint")

state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
missing, unexpected = context_encoder.load_state_dict(state_dict, strict=False)
print(f"  Missing keys:    {len(missing)}")
print(f"  Unexpected keys: {len(unexpected)}")

total_params = sum(p.numel() for p in context_encoder.parameters())
print(f"  Total encoder params: {total_params:,}")
print(f"  Transformer blocks:   {len(context_encoder.blocks) if hasattr(context_encoder, 'blocks') else 'N/A'}")


In [ ]:
# ── Cell 3: Create target encoder as deep copy ───────────────────────────────
import copy
target_encoder = copy.deepcopy(context_encoder)

# Target encoder is NEVER updated by gradient — only by EMA
for p in target_encoder.parameters():
    p.requires_grad = False

print("Target encoder created (EMA copy of context encoder)")
print(f"  All parameters frozen: {all(not p.requires_grad for p in target_encoder.parameters())}")


In [ ]:
# ── Cell 4: Build predictor (re-initialised from scratch) ────────────────────
predictor = IJEPAPredictor(
    encoder_embed_dim = VIT_EMBED_DIM,
    pred_embed_dim    = PRED_EMBED_DIM,
    num_patches       = NUM_PATCHES,
    num_heads         = PRED_NUM_HEADS,
    depth             = PRED_DEPTH,
    mlp_ratio         = PRED_MLP_RATIO,
    dropout           = PRED_DROPOUT,
)

pred_params = sum(p.numel() for p in predictor.parameters())
print(f"Predictor params: {pred_params:,}  ({pred_params/total_params*100:.2f}% of encoder)")
print(f"  Depth: {PRED_DEPTH} layers | Hidden: {PRED_EMBED_DIM} | Heads: {PRED_NUM_HEADS}")
print("  All predictor weights randomly initialised (NOT loaded from Meta checkpoint)")


In [ ]:
# ── Cell 5: Apply layer-wise LR & verify freeze strategy ─────────────────────
context_encoder = context_encoder.to(device)
target_encoder  = target_encoder.to(device)
predictor       = predictor.to(device)

optimizer = get_layerwise_optimizer(
    context_encoder, predictor,
    frozen_layers = FROZEN_LAYERS,
    low_lr_layers = LOW_LR_LAYERS,
    std_lr_layers = STD_LR_LAYERS,
    lr_head    = PT_LR_HEAD,
    lr_mid     = PT_LR_ENCODER_MID,
    lr_top     = PT_LR_ENCODER_TOP,
    weight_decay = PT_WEIGHT_DECAY,
)

# Verify frozen layers have zero trainable params
if hasattr(context_encoder, "blocks"):
    print("Layer freeze verification:")
    for i, block in enumerate(context_encoder.blocks):
        trainable = sum(p.numel() for p in block.parameters() if p.requires_grad)
        status = "FROZEN" if i in FROZEN_LAYERS else ("LOW LR" if i in LOW_LR_LAYERS else "STD LR")
        frozen_marker = "🔒" if i in FROZEN_LAYERS else "🔓"
        if i in FROZEN_LAYERS[:2] or i in [FROZEN_LAYERS[-1], LOW_LR_LAYERS[0],
                                             LOW_LR_LAYERS[-1], STD_LR_LAYERS[0], 31]:
            print(f"  Block {i:2d}: {status:8s} {frozen_marker} trainable={trainable:,}")
        elif i == 5:
            print("  ...")

total_trainable = sum(p.numel() for p in context_encoder.parameters() if p.requires_grad)
total_trainable += sum(p.numel() for p in predictor.parameters() if p.requires_grad)
print(f"\nTotal trainable params (encoder + predictor): {total_trainable:,}")
print(f"Frozen encoder params: {sum(p.numel() for p in context_encoder.parameters() if not p.requires_grad):,}")


In [ ]:
# ── Cell 6: Forward pass verification ─────────────────────────────────────────
print("Verifying forward pass shapes...")
test_imgs = torch.randn(4, 3, IMAGE_CROP, IMAGE_CROP).to(device)

with torch.no_grad():
    # Patch tokens from encoder
    tokens = context_encoder.forward_features(test_imgs)
    print(f"Encoder forward_features output: {tokens.shape}")
    # Should be (B, N_patches+1, D) where N_patches = 256 for ViT-H/14 at 224px
    assert tokens.dim() == 3, "Expected 3D output (B, N+1, D)"
    patch_tokens = tokens[:, 1:, :]  # drop CLS
    print(f"Patch tokens (after CLS drop): {patch_tokens.shape}")
    assert patch_tokens.shape == (4, NUM_PATCHES, VIT_EMBED_DIM), \
        f"Expected (4, {NUM_PATCHES}, {VIT_EMBED_DIM}), got {patch_tokens.shape}"

    # Predictor
    B, N_ctx = 4, 150
    ctx_emb  = torch.randn(B, N_ctx, VIT_EMBED_DIM).to(device)
    tgt_ids  = torch.randint(0, NUM_PATCHES, (B, 40)).to(device)
    pred_out = predictor(ctx_emb, tgt_ids)
    print(f"Predictor output: {pred_out.shape}  (expected (4, 40, {VIT_EMBED_DIM}))")
    assert pred_out.shape == (B, 40, VIT_EMBED_DIM)

print("\n✅ All shape checks passed.")


In [ ]:
# ── Cell 7: VRAM estimate ─────────────────────────────────────────────────────
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(device)
    test_imgs = torch.randn(PT_BATCH_SIZE, 3, IMAGE_CROP, IMAGE_CROP).to(device)
    with torch.cuda.amp.autocast():
        tokens = context_encoder.forward_features(test_imgs)
    peak_gb = torch.cuda.max_memory_allocated(device) / 1e9
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM at batch_size={PT_BATCH_SIZE}: {peak_gb:.2f} GB / {total_gb:.1f} GB")
    if peak_gb > total_gb * 0.85:
        print(f"⚠️  Close to VRAM limit. Reduce PT_BATCH_SIZE to {PT_BATCH_SIZE // 2}")
    else:
        print("✅ VRAM budget OK")
else:
    print("VRAM check skipped (CPU mode)")
print("\n✅ S4_1 Setup verification complete.")
